# Livewire v4 Documentation Crawler

This notebook crawls all documentation pages from [Livewire v4](https://livewire.laravel.com/docs/4.x/quickstart), saves raw HTML, converts content to Markdown, and generates a structured reference index.

**Workflow:**
1. Configuration
2. Fetch start page
3. Extract sidebar menu links
4. Crawl each page (with caching)
5. Convert HTML to Markdown
6. Generate references.md index

## Library Imports

In [1]:
import os
import re
import time
import random
import json
import warnings
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup
from tqdm import tqdm
from html_to_markdown import convert, ConversionOptions

warnings.filterwarnings('ignore')
print('All libraries loaded successfully.')

All libraries loaded successfully.


## Configuration

In [2]:
NAME = 'livewire-v4'
BASE_URL = 'https://livewire.laravel.com'
START_URL = 'https://livewire.laravel.com/docs/4.x/quickstart'

BASE_DIR = f"docs/{NAME.replace('-', '_')}"
HTML_DIR = f"{BASE_DIR}/html"
REFERENCES_DIR = f"{BASE_DIR}/references"
REFERENCES_JSON = os.path.join(BASE_DIR, 'references.json')

DELAY_RANGE = (0.2, 0.6)
MAX_AGE_DAYS = 7
MAX_AGE_SECONDS = MAX_AGE_DAYS * 86400

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.9',
}

SIDEBAR_SELECTOR = 'aside'
CONTENT_SELECTOR = 'div.docsearch-content'

print(f'NAME: {NAME}')
print(f'BASE_URL: {BASE_URL}')
print(f'START_URL: {START_URL}')
print(f'BASE_DIR: {BASE_DIR}')
print(f'HTML_DIR: {HTML_DIR}')
print(f'REFERENCES_DIR: {REFERENCES_DIR}')
print(f'REFERENCES_JSON: {REFERENCES_JSON}')
print(f'DELAY_RANGE: {DELAY_RANGE}')
print(f'MAX_AGE_DAYS: {MAX_AGE_DAYS}')
print(f'SIDEBAR_SELECTOR: {SIDEBAR_SELECTOR}')
print(f'CONTENT_SELECTOR: {CONTENT_SELECTOR}')

os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(HTML_DIR, exist_ok=True)
os.makedirs(REFERENCES_DIR, exist_ok=True)
print('\nDirectories created successfully.')

NAME: livewire-v4
BASE_URL: https://livewire.laravel.com
START_URL: https://livewire.laravel.com/docs/4.x/quickstart
BASE_DIR: docs/livewire_v4
HTML_DIR: docs/livewire_v4/html
REFERENCES_DIR: docs/livewire_v4/references
REFERENCES_JSON: docs/livewire_v4\references.json
DELAY_RANGE: (0.2, 0.6)
MAX_AGE_DAYS: 7
SIDEBAR_SELECTOR: aside
CONTENT_SELECTOR: div.docsearch-content

Directories created successfully.


## Step 2: Fetch Start Page

Fetch the starting documentation page and parse it with BeautifulSoup to prepare for sidebar link extraction.

In [3]:
print(f'Fetching: {START_URL}')
response = requests.get(START_URL, headers=HEADERS, timeout=30)
response.raise_for_status()

soup = BeautifulSoup(response.text, 'html.parser')
print(f'Page fetched successfully. Size: {len(response.text)} bytes')
print(f'Title: {soup.title.string.strip() if soup.title else "N/A"}')

Fetching: https://livewire.laravel.com/docs/4.x/quickstart
Page fetched successfully. Size: 229867 bytes
Title: Quickstart | Laravel Livewire


## Step 3: Extract Sidebar Menu Links

Find the sidebar navigation container using the inspected selector, extract all documentation links, deduplicate, and save to `references.json`.

In [4]:
sidebar = soup.select_one(SIDEBAR_SELECTOR)
if not sidebar:
    raise ValueError(f'Sidebar selector "{SIDEBAR_SELECTOR}" not found. Check the site structure.')

links_data = []
seen_urls = set()

for a_tag in sidebar.find_all('a', href=True):
    href = a_tag['href']
    if '/docs/4.x/' not in href:
        continue
    full_url = urljoin(BASE_URL, href)
    if full_url in seen_urls:
        continue
    seen_urls.add(full_url)

    title = a_tag.get_text(strip=True)
    if not title:
        continue

    slug = title.lower()
    slug = re.sub(r'[^a-z0-9]+', '-', slug).strip('-')
    slug = slug[:80]

    links_data.append({
        'title': title,
        'slug': slug,
        'url': full_url,
    })

with open(REFERENCES_JSON, 'w', encoding='utf-8') as f:
    json.dump(links_data, f, indent=2, ensure_ascii=False)

print(f'Found {len(links_data)} documentation links.')
print(f'Saved to: {REFERENCES_JSON}')
print()
print('First 5 entries:')
for entry in links_data[:5]:
    print(f'  [{entry["slug"]}] {entry["title"]} -> {entry["url"]}')

Found 80 documentation links.
Saved to: docs/livewire_v4\references.json

First 5 entries:
  [version-4-x] Version 4.x -> https://livewire.laravel.com/docs/4.x/quickstart
  [installation] Installation -> https://livewire.laravel.com/docs/4.x/installation
  [upgrade-guide] Upgrade Guide -> https://livewire.laravel.com/docs/4.x/upgrading
  [components] Components -> https://livewire.laravel.com/docs/4.x/components
  [pages] Pages -> https://livewire.laravel.com/docs/4.x/pages


## Step 4: Crawl Each Page

Read the `references.json` file and download each page with caching and random delays to avoid rate-limiting.

In [5]:
with open(REFERENCES_JSON, 'r', encoding='utf-8') as f:
    links_data = json.load(f)

errors = []
done = 0
skipped = 0

for idx, entry in enumerate(tqdm(links_data, desc='Crawling pages')):
    html_path = os.path.join(HTML_DIR, f'{idx+1:03d}_{entry["slug"]}.html')

    if os.path.exists(html_path):
        age = time.time() - os.path.getmtime(html_path)
        if age < MAX_AGE_SECONDS:
            skipped += 1
            continue

    try:
        resp = requests.get(entry['url'], headers=HEADERS, timeout=30)
        resp.raise_for_status()
        with open(html_path, 'w', encoding='utf-8') as f:
            f.write(resp.text)
        done += 1
        time.sleep(random.uniform(*DELAY_RANGE))
    except Exception as e:
        errors.append(f'{entry["title"]} ({entry["url"]}): {e}')

print(f'\nDownload complete: {done} new, {skipped} cached, {len(errors)} failed')
if errors:
    print('\nErrors:')
    for err in errors:
        print(f'  - {err}')

Crawling pages: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 80/80 [00:00<00:00, 9077.35it/s]


Download complete: 0 new, 80 cached, 0 failed


## Step 5: Convert HTML to Markdown

Read each saved HTML file, extract the content area, remove image tags, and convert to clean Markdown.

In [6]:
with open(REFERENCES_JSON, 'r', encoding='utf-8') as f:
    links_data = json.load(f)

errors = []

for idx, entry in enumerate(tqdm(links_data, desc='Converting to Markdown')):
    html_path = os.path.join(HTML_DIR, f'{idx+1:03d}_{entry["slug"]}.html')
    md_path = os.path.join(REFERENCES_DIR, f'{idx+1:03d}_{entry["slug"]}.md')

    if not os.path.exists(html_path):
        errors.append(f'HTML file not found: {html_path}')
        continue

    with open(html_path, 'r', encoding='utf-8') as f:
        html_content = f.read()

    page_soup = BeautifulSoup(html_content, 'html.parser')
    content_el = page_soup.select_one(CONTENT_SELECTOR)

    if not content_el:
        errors.append(f'Content selector not found in: {entry["title"]}')
        continue

    for tag in content_el.find_all(['img', 'svg', 'picture', 'source']):
        tag.decompose()

    html_str = str(content_el)

    options = ConversionOptions(heading_style='atx', wrap=False)
    markdown_text = convert(html_str, options)

    with open(md_path, 'w', encoding='utf-8') as f:
        f.write(markdown_text)

print(f'\nConversion complete. {len(links_data) - len(errors)} converted, {len(errors)} errors')
if errors:
    print('\nErrors:')
    for err in errors:
        print(f'  - {err}')

Converting to Markdown: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 80/80 [00:20<00:00,  3.87it/s]


Conversion complete. 80 converted, 0 errors


## Step 6: Generate references.md Index

Read the `references.json` metadata and build a structured index file with section groupings.

In [7]:
with open(REFERENCES_JSON, 'r', encoding='utf-8') as f:
    links_data = json.load(f)

lines = []
lines.append('# Livewire v4 Documentation References')
lines.append('')
lines.append(f'Total pages: {len(links_data)}')
lines.append('')

for entry in links_data:
    lines.append(f'- [{entry["title"]}](references/{entry["slug"]}.md)')

index_path = os.path.join(BASE_DIR, 'references.md')
with open(index_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))

print(f'Index saved to: {index_path}')
print(f'Total entries: {len(links_data)}')

Index saved to: docs/livewire_v4\references.md
Total entries: 80


## Summary

The crawler has completed all steps:

1. **Configuration** - Set up all parameters and selectors
2. **Fetch start page** - Retrieved the starting documentation page
3. **Extract sidebar links** - Saved all documentation URLs to `references.json`
4. **Crawl pages** - Downloaded each page with caching to `html/`
5. **Convert to Markdown** - Converted content to clean `.md` files in `references/`
6. **Generate index** - Created `references.md` as a structured entry point

### Output Structure

```
docs/livewire_v4/
├── references.json    ← Link metadata
├── references.md      ← Index file
├── references/        ← Individual Markdown files
│   ├── 001_quickstart.md
│   ├── 002_installation.md
│   └── ...
└── html/              ← Raw HTML files (temporary)
    ├── 001_quickstart.html
    ├── 002_installation.html
    └── ...
```

Next step: AI refines `references.md` with descriptions, section groupings, and context.